In [ ]:
!pip install opencv-python

In [3]:
%pip install "numpy<2.0"

  Using cached numpy-1.26.4-cp311-cp311-win_amd64.whl (15.8 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.6
    Uninstalling numpy-2.2.6:
      Successfully uninstalled numpy-2.2.6
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'C:\\Users\\solvi\\anaconda3\\Lib\\site-packages\\~umpy.libs\\libscipy_openblas64_-13e2df515630b4a41f92893938845698.dll'
Consider using the `--user` option or check the permissions.



In [ ]:
import pandas as pd
import os
import shutil
from sklearn.model_selection import train_test_split


csv_path = 'data.csv'
img_source_dir = 'images/'
output_dir = 'yolo_dataset/'
class_mapping = {'WBC': 0, 'RBC': 1, 'Platelets': 2}
#bc no header
df = pd.read_csv(csv_path, header=None, names=['filename', 'class', 'xmin', 'ymin', 'xmax', 'ymax'])

for split in ['train', 'test']:
    os.makedirs(os.path.join(output_dir, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(output_dir, 'labels', split), exist_ok=True)

unique_files = df['filename'].unique()
train_files, test_files = train_test_split(unique_files, test_size=0.2, random_state=42)

def convert_and_save(filenames, split_name):
    print(f"Processing {len(filenames)} images for {split_name}...")
    
    for filename in filenames:
        src_img = os.path.join(img_source_dir, filename)
        dst_img = os.path.join(output_dir, 'images', split_name, filename)

        if not os.path.exists(src_img):
            print(f"Warning: {filename} not found in images folder.")
            continue
            
        shutil.copy(src_img, dst_img)
        import cv2
        img = cv2.imread(src_img)
        if img is None: continue
        img_h, img_w = img.shape[:2]
        
        img_df = df[df['filename'] == filename]
        txt_filename = filename.replace('.jpg', '.txt')
        txt_path = os.path.join(output_dir, 'labels', split_name, txt_filename)
        
        with open(txt_path, 'w') as f:
            for _, row in img_df.iterrows():
                cls_name = row['class']
                if cls_name not in class_mapping: continue
                
                cls_id = class_mapping[cls_name]
                xmin, ymin = row['xmin'], row['ymin']
                xmax, ymax = row['xmax'], row['ymax']
                
                # YOLO MATH: Normalize coordinates (0-1)
                box_w = xmax - xmin
                box_h = ymax - ymin
                box_cx = xmin + (box_w / 2)
                box_cy = ymin + (box_h / 2)
                
                #:by dividing by image width/height
                norm_cx = box_cx / img_w
                norm_cy = box_cy / img_h
                norm_w = box_w / img_w
                norm_h = box_h / img_h
                #<class_id> <x_center> <y_center> <width> <height>
                f.write(f"{cls_id} {norm_cx:.6f} {norm_cy:.6f} {norm_w:.6f} {norm_h:.6f}\n")

convert_and_save(train_files, 'train')
convert_and_save(test_files, 'test')
print("Conversion and Split Complete! Folder 'yolo_dataset' is ready for training.")

c:\Users\solvi\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Processing 291 images for train...
Processing 73 images for test...
Conversion and Split Complete! Folder 'yolo_dataset' is ready for training.
